In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="bronze_claims",
    comment="Raw claims ingested from volume",
    table_properties={"quality": "bronze"}
)
def bronze_claims():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .load("/Volumes/health_care_2026/healthcare_fraud_demo/inputs/")
    )

In [0]:
@dlt.table(
    name="silver_claims",
    comment="Cleaned claims",
    table_properties={"quality": "silver"}
)
@dlt.expect("valid_amount", "amount > 0")
def silver_claims():
    df = dlt.read("bronze_claims")

    return (
        df.withColumn("amount", col("amount").cast("double"))
          .filter(col("claim_id").isNotNull())
    )

In [0]:
@dlt.table(
    name="gold_claim_summary",
    comment="Claim summary by status",
    table_properties={"quality": "gold"}
)
def gold_claim_summary():
    df = dlt.read("silver_claims")

    return (
        df.groupBy("status")
          .agg(sum("amount").alias("total_amount"))
    )